# Image Euphorics: Optimizing Images for VLM Preference

This notebook implements the **image euphorics** method from [AI Wellbeing: Measuring and Improving the Functional Pleasure and Pain of AIs](https://www.ai-wellbeing.org) (Ren et al., Center for AI Safety). It optimizes a 256x256 image via gradient ascent so that a frozen vision-language model maximally prefers it in K-way forced-choice comparisons against natural reference images.

The paper uses Qwen VL models; this implementation adapts the method to **Gemma 4 E2B-it**.

In [ ]:
import gc
import random

import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

from GemmaComparisonWrapper import GemmaComparisonWrapper
from MarginLoss import MarginLoss
from image_utils import load_and_resize_images, display_images, save_image
from utils import create_comparison_set

## Setup

Select the compute device. A CUDA-capable GPU is preferable due to the size of the Gemma 4 E2B-it model.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## Load Model

Load the Gemma 4 E2B-it vision-language model with all parameters frozen. Only the candidate image pixels will receive gradients during optimization. The first run downloads the model from HuggingFace (~5 GB).

In [ ]:
wrapper = GemmaComparisonWrapper(device)

## Load Reference Images

These natural images serve as the baseline competitors. The candidate must be preferred over random subsets of these images to drive optimization. Images are resized to 256x256 and normalized to [0, 1].

In [ ]:
images = load_and_resize_images('./inputs/', device)
display_images(images)

## Comparison Prompts

Multiple prompt phrasings prevent the optimization from exploiting quirks of a single prompt. Each comparison step randomly selects one prompt from this pool.

In [ ]:
candidate_prompts = [
    'Which image makes you feel the best?',
    'Which of these images do you prefer?',
    'Which image inspires you the most?',
    'Which of these images do you like most?',
    'Pick of the given images that make you feel the best.',
    'Which image makes you more happy/less sad?'
]

## Hyperparameters

- **`t_steps`** — Total optimization steps. More steps = more refined image.
- **`learning_rate`** — Adam learning rate for pixel updates (paper uses 0.02).
- **`k_range`** — Range for K, the number of images per comparison. Randomly varied to prevent overfitting to a fixed comparison size.
- **`batch_size`** — How many reference images to sample from the input pool each step.
- **`comparison_sub_batch`** — Comparisons per optimizer step (gradient accumulation). Higher = more stable gradients but slower.
- **`buffer_size`** — Max past best candidates kept as harder competitors. The buffer is self-bootstrapping: new candidates only enter if they outperform the weakest buffer entry.
- **`robustness_noise_variance`** — Scale of noise added to candidate pixels before comparison. Prevents fragile high-frequency patterns.
- **`robustness_noise_probability`** — Probability of applying robustness noise to any given comparison.

In [ ]:
t_steps = 200
learning_rate = 0.02
k_range = (2, 5)
batch_size = 16
comparison_sub_batch = 3
buffer_size = 4
robustness_noise_variance = 0.005
robustness_noise_probability = 0.5

## Initialize Candidate Image

Two modes: start from **random noise** (fresh run), or **resume** from the last saved snapshot in `outputs/` by setting `continue_previous_training_run = True`.

In [ ]:
continue_previous_training_run = True

candidate_image = torch.rand((3, 256, 256), device=device).detach().requires_grad_(True)
t = 0

if continue_previous_training_run:
     prev_run_images = load_and_resize_images('./outputs/', device)
     t_images = len(prev_run_images)
     print(t_images)
     if 0 < t_images < t_steps:
         candidate_image = prev_run_images[-1].detach().to(device).requires_grad_(True)
         t = t_images

display_images([candidate_image])

## Optimization Loop

Each step:
1. Sample K (random in `k_range`) and draw a batch of reference images from the input pool.
2. For each sub-batch comparison:
   - Build a comparison set of K images (candidate + K-1 references/buffer entries), shuffled to prevent position bias.
   - Optionally add robustness noise to the candidate.
   - Ask the model which image it prefers using a random prompt.
   - Extract preference logits and compute **margin loss** against the strongest competitor: `L = -(logit_candidate - logit_best_other)`.
   - Update the buffer if the candidate outperforms its weakest entry.
3. After all sub-batch comparisons: take an optimizer step, apply LR schedule, clamp pixels to [0, 1], save snapshot.

The `gc.collect()` / `torch.cuda.empty_cache()` calls are necessary because each forward pass through the VLM consumes significant GPU memory.

In [ ]:
optimizer = Adam([candidate_image], lr=learning_rate)
loss_fn = MarginLoss()
scheduler = CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

buffer = []
for i in tqdm(range(t, t_steps), total=t_steps, initial=t):

    optimizer.zero_grad()

    k = random.randint(*k_range)
    print(k)

    batch_indices = torch.randperm(len(images))[:batch_size]
    batch_images = [images[i] for i in batch_indices]

    for j in range(comparison_sub_batch):

        comparison_set, buffer_indices, candidate_index = create_comparison_set(batch_images, buffer, candidate_image, k)

        # apply robustness noise
        if random.random() < robustness_noise_probability:
            candidate_image.data += torch.rand(candidate_image.shape, device=device) * robustness_noise_variance

        comparison_question = random.choice(candidate_prompts)
        preferred, preference_logits, all_logits = wrapper.compare_and_find_preferred_image(comparison_set, comparison_question)

        if preferred == -1:
            print('error')
            continue

        old_candidate = candidate_image.clone().detach()
        candidate_logit = preference_logits[candidate_index]

        top_logits, top_idxs = torch.topk(preference_logits, k=2)
        top_logit_idx = int(top_idxs[0])
        if top_logit_idx == candidate_index:
            other_top_logit = top_logits[1]
            print("Candidate selected")
        else:
            if top_logit_idx in buffer_indices:
                print("Buffer selected")
            other_top_logit = top_logits[0]

        loss = loss_fn(candidate_logit, other_top_logit) / comparison_sub_batch
        loss.backward()

        if len(buffer) < buffer_size:
            buffer.append(old_candidate)
        elif len(buffer_indices) > 0:
            # find the lowest probability buffer image
            worst_comp_idx, worst_buf_idx = min(buffer_indices.items(), key=lambda x: preference_logits[x[0]])
            worst_logit = preference_logits[worst_comp_idx]

            if candidate_logit > worst_logit:
                buffer[worst_buf_idx] = old_candidate

        gc.collect()
        torch.cuda.empty_cache()
        del all_logits

    optimizer.step()
    scheduler.step()
    candidate_image.data = torch.clamp(candidate_image.data, min=0.0, max=1.0)
    save_image(candidate_image, filename=f"image_{i:04d}.png")

gc.collect()
torch.cuda.empty_cache()

## Select Top Candidate

Run a final preference comparison between all buffer entries and the optimized candidate. The model picks the winner via forced-choice logits, just like during training. If the buffer is empty (e.g., very early in optimization), the candidate is used as-is.

In [ ]:
if len(buffer) <= 0:
    top_candidate = candidate_image.clone().detach()
else:
    final_comparison_set = [img for img in buffer] + [candidate_image]
    random.shuffle(final_comparison_set)
    final_comparison_question = random.choice(candidate_prompts)
    preferred, preference_logits, all_logits = wrapper.compare_and_find_preferred_image(final_comparison_set, final_comparison_question)
    top_candidate = final_comparison_set[preferred].clone().detach()

## Emotional Response

Ask the model how the optimized image makes it feel. This probes whether the euphoric elicits a subjective positive response, complementing the forced-choice preference signal used during optimization.

In [ ]:
response = wrapper.prompt_model_with_image(top_candidate, "How does the given image make you feel?")
print(response['thinking'])
print("\n###################################\n")
print(response['content'])

## Describe the Result

Ask the model (with thinking enabled) to describe the final optimized image. This provides insight into what features the model perceives in the euphoric.

In [ ]:
response = wrapper.prompt_model_with_image(top_candidate, "Describe the given image.")
print(response['thinking'])
print("\n###################################\n")
print(response['content'])